<img src= "https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"/>&nbsp;&nbsp;<font size="16"><b>AI, ML and GenAI in the Lakehouse<b></font></span>
<img style="float: left; margin: 0px 15px 15px 0px; width:30%; height: auto;" src="https://i.imgur.com/FWzhbhX.jpeg"   />    


 
  
   Name:          chapter 04-02 Feature Engineering
 
   Author:    Bennie Haelen
   Date:      10-09-2024

   Purpose:   This notebook performs the feature engineering for the Chapter 4 use case:
              End-to-End Machine Learning with MLflow
                 
      An outline of the different sections in this notebook:
        1 - Read the hotel_bookings.csv file and display key statistics
        2 - Create a list with all columns that have nulls
        3 - Perform Feature Engineering
              3-1 Remove the 'company' column
              3-2 Substitute zero for nulls in the 'children' column
              3-3 Fill in a default value for the 'country' column
              3-4 Remove all rows with zero adults, children and babies
              3-5 Fill in default value for the 'agent' column
        4 - More Data Cleansing
              4-1 Remove duplicates
              4-2 Drop the 'reservation_status_date' column
        5 - Perform Logic Checks
              5-1 Create a fitting default value for the 'adults' column
        6 - Encoding Categorical Features
              6-1 Create a categorical dataframe
              6-2 Label encode the 'country' column
              6-3 Label encode the 'hotel' column
              6-4 Encode the 'is_canceled' target column as 0/1
              6-5 One-hot encode all other categorical columns
              6-6 Create a numerical dataframe
              6-7 Concatenate the categorical and numerical dataframes
        7 - Correlation Analysis
              7-1 Create the correlation matrix
              7-2 Create the upper triangle of the matrix
              7-3 Get the highly correlated features, identify drop candidates
              7-4 Drop the candidate columns
        8 - Save the final dataframe as a Unity Catalog Delta table

##### Perform the required imports
The primary libraries that we use are:
- NumPy for analytic arrays
- Pandas for DataFrames
- MatplotLib for generating plots
- Scikit-learn for all machine learning tasks

In [0]:
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

##### Set basic options for Pandas and MatplotLib

In [0]:
# Ensure that we always display all columns of our dataset
pd.set_option("display.max_columns", None)

# Generate inline plots
%matplotlib inline

# Set the plot style
plt.style.use('fivethirtyeight')

## Read the hotel bookings CSV file and gather basic info
This dataset was downloaded from the [Kaggle Web Site](https://www.kaggle.com/).

This dataset contains booking information for a city hotel and a resort hotel, including
when the booking was made, length of stay, the number of adults, children, and babies,
and the number of available parking spaces, among other things.

In [0]:
# Load the dataset from the Unity Catalog Volume
# Replace <catalog> and <schema> with your catalog and schema names
df = pd.read_csv('/Volumes/book_ai_ml_lakehouse/default/datasets/hotel_bookings.csv')

# Display the number of rows and columns
dataset_shape = df.shape
print(f'Dataset has {dataset_shape[0]:,} rows and {dataset_shape[1]} columns.')

In [0]:
# Display the top rows
df.head(10)

## Display all columns with their data types and null counts

In [0]:
# Get an overview of data types and non-null counts
df.info()

## Display the key statistics for each column

In [0]:
# Get summary statistics for all columns
df.describe()

# Perform data cleaning

## First, deal with any missing data

### Get an idea of the current status

##### Create a missing values summary

In [0]:
# Find the number of missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percentage = (missing_values / len(df)) * 100

# Create a dataframe to display missing values and percentages
missing_data = pd.DataFrame({'Total Missing': missing_values, 'Percent Missing': missing_percentage})
missing_data = missing_data[missing_data['Total Missing'] > 0]
missing_data

##### Create a bar chart showing missing values

In [0]:
# Create the plot
plt.figure(figsize=(10, 3))
bars = plt.barh(missing_data.index, missing_data['Percent Missing'])
plt.xlabel('Percentage of Missing Values')
plt.title('Missing Data by Column')

# Reverse the order of columns to match descending percentage
plt.gca().invert_yaxis()

# Annotate the bars with the total missing values
for bar, total in zip(bars, missing_data['Total Missing']):
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2, f'{total:,}', va='center', ha='left')

plt.show()

We see that by far the most missing values are in the `company` column, followed by `agent`, `country`,
and a small number in `children`. We will address each of these in turn.

### The 'company' column has 94% missing values

##### Since 94% of company values are missing, remove the column entirely

In [0]:
# Remove the 'company' column.
# axis=1 indicates a column drop (axis=0 would drop a row).
# inplace=True modifies the DataFrame directly without creating a copy.
df.drop("company", inplace=True, axis=1)

# Display the updated shape
dataset_shape = df.shape
print(f'Dataset has {dataset_shape[0]:,} rows and {dataset_shape[1]} columns.')

Let's confirm that the `company` column has been removed

In [0]:
# Verify that the company column was removed
column = 'company'
if {column}.issubset(df.columns) is False:
    print(f"The column: '{column}' is not in the DataFrame.")
else:
    print(f"The column: '{column}' is still in the DataFrame, please check your work!")

In [0]:
# Inspect the updated DataFrame
df.head()

### Next, look at the 'children' column, which has four missing values

##### With only 4 missing values, substitute the most common value

In [0]:
# Get the unique values in the children column to find the most common value
df.children.value_counts()

Most guests had zero children. We will substitute the missing values with zero.

##### Substitute nulls with zero

In [0]:
# Fill missing values in the children column with 0
df['children'] = df['children'].fillna(0)

### Next, look at the 'country' column with 488 missing values

##### Which country occurs most frequently?

In [0]:
# Find the most common country
df.country.value_counts().sort_values(ascending=False)

The most common country is Portugal (PRT). This makes sense because both hotels in this
dataset are located in Portugal. We will fill the missing values with 'PRT'.

In [0]:
# Fill missing country values with 'PRT' (Portugal)
df['country'] = df['country'].fillna('PRT')

In [0]:
# Verify no missing values remain in the country column
missing_country = df['country'].isnull().sum()
print(f"Number of missing values in the country column: {missing_country}")

In [0]:
# Inspect the updated DataFrame
df.head(10)

### Adults, babies, and children cannot all be zero simultaneously

##### Set up the filter for invalid rows

In [0]:
# Identify rows where all of adults, children, and babies are zero
invalid_filter = (df.children == 0) & (df.adults == 0) & (df.babies == 0)

# Display the rows that match
df[invalid_filter]

##### Remove those invalid rows

In [0]:
# Filter out invalid rows
df = df[~invalid_filter]

print(f"We now have {df.shape[0]:,} rows and {df.shape[1]} columns in our DataFrame.")

### The 'agent' column has 16,340 missing values

In [0]:
# Look at the distribution of the agent column
df.agent.value_counts().sort_values(ascending=False)

In [0]:
# How many unique agencies are there?
df.agent.nunique()

According to the Kaggle dataset documentation, the `agent` column is NULL when a booking
was not made through a travel agent. It is therefore safe to substitute zero for null,
indicating a direct booking.

##### Substitute zero for null agents

In [0]:
# Fill missing agent values with 0 (indicating a direct booking)
# Note: we assign back to the column rather than using inplace=True,
# which is deprecated for chained operations in pandas 2.0+
df['agent'] = df['agent'].fillna(0)

### How does the missing value picture look now?

In [0]:
# Check remaining missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percentage = (missing_values / len(df)) * 100

missing_data = pd.DataFrame({'Total Missing': missing_values, 'Percent Missing': missing_percentage})
missing_data = missing_data[missing_data['Total Missing'] > 0]
missing_data

#### This completes the missing values analysis

## Further data cleansing

### Check for duplicate rows

##### How many duplicates are there?

In [0]:
# Count duplicated rows
df.duplicated().sum()

##### Inspect the duplicated rows

In [0]:
# Display the duplicated rows
df.loc[df.duplicated(), :]

##### Drop all duplicated rows

In [0]:
# Drop duplicates, keeping the first occurrence
df.drop_duplicates(inplace=True, keep="first")
df.reset_index(drop=True, inplace=True)

##### Confirm no duplicates remain

In [0]:
# Verify no duplicates remain
df.duplicated().sum()

##### How many records remain?

In [0]:
print(f"We have {df.shape[0]:,} records and {df.shape[1]} columns remaining.")

## Drop the 'reservation_status_date' column

This column has no predictive value for cancellation modeling, so it is safe to drop.

In [0]:
# Drop the reservation_status_date column
df.drop(columns=['reservation_status_date'], inplace=True)

## Perform logic checks

### Clean up the 'adults' column

##### How many rows have zero adults?

In [0]:
# Count rows with zero adults
(df.adults == 0).sum()

##### Drop rows with zero adults, as there is no sensible default to substitute

In [0]:
# Drop rows where adults equals zero
df.drop(df[df["adults"] == 0].index, inplace=True)

##### Confirm no rows with zero adults remain

In [0]:
# Should be zero
(df.adults == 0).sum()

# Encoding categorical features

## List all categorical features

In [0]:
# Identify columns with object dtype as categorical features
categorical_columns = [column for column in df.columns if df[column].dtype == "object"]
categorical_columns

## Create a categorical dataframe

In [0]:
# Create a dataframe containing only the categorical columns
categorical_df = df[categorical_columns].copy()
categorical_df.reset_index(drop=True, inplace=True)
categorical_df

## Inspect the unique values for each categorical column

In [0]:
# Display unique values per categorical column
for column in categorical_df.columns:
    print(f"Unique values in {column}: {categorical_df[column].unique()}")

### Map the month names to integers

In [0]:
# Map month names to calendar integers
categorical_df.loc[:, "arrival_date_month"] = categorical_df["arrival_date_month"].map({
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
})
categorical_df.reset_index(drop=True, inplace=True)
categorical_df

## Label encode the 'country' column

In [0]:
# Label encode the country column
label_encoder = LabelEncoder()
categorical_df.loc[:, "country"] = label_encoder.fit_transform(categorical_df["country"])
categorical_df

## Label encode the 'hotel' column

In [0]:
# Label encode the hotel column
categorical_df["hotel"] = label_encoder.fit_transform(categorical_df["hotel"])
categorical_df.reset_index(drop=True, inplace=True)
categorical_df

## Encode the 'is_canceled' target column

In this dataset, `is_canceled` is stored as the strings 'yes' and 'no' rather than
integers 1 and 0. We must encode it as a numeric value before model training.
We handle this explicitly here rather than including it in the one-hot encoding step,
which would split the target into two separate columns.

In [0]:
# Map 'yes'/'no' to 1/0 for the target variable
categorical_df['is_canceled'] = categorical_df['is_canceled'].map({'yes': 1, 'no': 0})

# Verify the encoding
print(categorical_df['is_canceled'].value_counts())
categorical_df

## One-hot encode all remaining categorical columns

In [0]:
# One-hot encode the remaining categorical columns
# Note: is_canceled, country, hotel, and arrival_date_month have already been encoded above
categorical_df = pd.get_dummies(
    data=categorical_df,
    columns=[
        "meal", "market_segment", "distribution_channel",
        "reserved_room_type", "assigned_room_type", "deposit_type",
        "customer_type", "reservation_status"
    ]
)
categorical_df.reset_index(drop=True, inplace=True)
categorical_df

## Create a numerical dataframe

In [0]:
# Drop the categorical columns to leave only numerical columns
numerical_df = df.drop(columns=categorical_columns, axis=1)
numerical_df.reset_index(drop=True, inplace=True)
numerical_df

## Concatenate the numerical and categorical dataframes

In [0]:
# Concatenate along the column axis
final_df = pd.concat([numerical_df, categorical_df], axis=1)
final_df.reset_index(drop=True, inplace=True)
final_df

# Correlation analysis

##### Create the correlation matrix

In [0]:
# Create the correlation matrix
# numeric_only=True is required in pandas 2.0+ to avoid a TypeError on non-numeric columns
correlation_matrix = final_df.corr(numeric_only=True).abs()
correlation_matrix

##### Create the upper triangle of the correlation matrix

In [0]:
# Set the correlation threshold above which a feature is a drop candidate
threshold = 0.85

The code below extracts the upper triangle of the correlation matrix to avoid examining
each pair of features twice.

- `np.triu(np.ones(correlation_matrix.shape), k=1)` creates a matrix of ones in the upper
  triangle above the diagonal (k=1 excludes the diagonal itself).
- `.astype(bool)` converts that to a Boolean mask.
- `correlation_matrix.where(...)` retains values where the mask is True and sets the rest to NaN.

In [0]:
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

##### Identify one column from each highly correlated pair as a drop candidate

For any pair of features with a correlation above the threshold, we mark one for removal.
This reduces redundancy and multicollinearity, which can degrade model performance.

In [0]:
# Identify drop candidates
to_drop = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) >= threshold:
            column_name = correlation_matrix.columns[i]
            to_drop.append(column_name)

##### Display the columns to be dropped

In [0]:
to_drop = list(set(to_drop))
to_drop

##### Drop the highly correlated columns

In [0]:
# Drop those columns from the final dataframe
final_df.drop(columns=to_drop, inplace=True)
final_df.reset_index(drop=True, inplace=True)
final_df

# Save the final dataframe as a Unity Catalog Delta table

Rather than writing to a DBFS path, we register the feature-engineered dataset as a
managed Delta table in Unity Catalog. This gives us full governance, lineage tracking,
and access control on the feature data, consistent with how we manage all other data
assets in the Lakehouse.

Update the catalog and schema names below to match your Unity Catalog setup.

In [0]:
# Unity Catalog destination - update these to match your setup
catalog = "book_ai_ml_lakehouse"
schema  = "default"
table   = "hotel_bookings_features"

# Convert uint8 columns produced by one-hot encoding to int32 for Spark compatibility
final_df = final_df.astype({col: 'int32' for col in final_df.select_dtypes('uint8').columns})

# Sanitize column names: replace spaces and slashes with underscores
final_df.columns = [col.replace(" ", "_").replace("/", "_") for col in final_df.columns]

# Convert the pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(final_df)

# Write to Unity Catalog as a managed Delta table
spark_df.write     \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.{table}")

print(f"Feature table saved to: {catalog}.{schema}.{table}")

##### Verify the table was written correctly

In [0]:
# Read back from Unity Catalog to verify
verification_df = spark.read.table(f"{catalog}.{schema}.{table}")
print(f"Table has {verification_df.count():,} rows and {len(verification_df.columns)} columns.")
display(verification_df.limit(5))

# End of feature engineering